In [2]:
import sqlite3
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load data
conn = sqlite3.connect("/Users/Maazuddin/Downloads/inventory.db")

df = pd.read_sql_query(
    """
    SELECT Dollars, Quantity, Freight
    FROM vendor_invoice
    """,
    conn
)

conn.close()

# Keep valid rows for rate calculation
df = df.dropna(subset=["Dollars", "Quantity", "Freight"])
df = df[df["Dollars"] > 0].copy()

# Target: freight rate
df["Freight_Rate"] = df["Freight"] / df["Dollars"]

X = df[["Dollars", "Quantity"]]
y = df["Freight"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Existing direct-freight model
direct_model = LinearRegression()
direct_model.fit(X_train, y_train)

direct_predictions = direct_model.predict(X_test)

# Rate model: train only on training rows
train_rates = (
    df.loc[X_train.index, "Freight"]
    / df.loc[X_train.index, "Dollars"]
)

rate_model = LinearRegression()
rate_model.fit(X_train[["Dollars"]], train_rates)

predicted_rates = rate_model.predict(X_test[["Dollars"]])

# Convert predicted rates back to freight amounts
rate_predictions = (
    predicted_rates * X_test["Dollars"].to_numpy()
)

def evaluate(name, actual, predicted):
    print(f"\n{name}")
    print(f"MAE:  {mean_absolute_error(actual, predicted):.4f}")
    print(
        f"RMSE: {np.sqrt(mean_squared_error(actual, predicted)):.4f}"
    )
    print(f"R²:   {r2_score(actual, predicted):.4f}")

evaluate("Direct Freight Model", y_test, direct_predictions)
evaluate("Freight Rate Model", y_test, rate_predictions)


Direct Freight Model
MAE:  24.4592
RMSE: 124.4288
R²:   0.9700

Freight Rate Model
MAE:  28.7166
RMSE: 150.5703
R²:   0.9561
